In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from dataclasses import dataclass
from sklearn.metrics import mean_squared_error
import lightgbm as lgb

SEED = 42
TRAIN_PATH = 'train_clean.csv'
SUB_PATH = 'sample_submission.csv'
HORIZON = 56
VAL_DAYS = 28

FAST_MODE = True
USE_RF = False
USE_XGB = False

np.random.seed(SEED)
print('Config loaded. FAST_MODE=', FAST_MODE)


In [ ]:
print('[1] Load and clean input...')
float_cols = ['Quantity', 'UnitPrice', 'SalesAmount', 'Unit Cost', 'Cost Amount']
usecols = ['Date', 'ItemCode'] + float_cols

df_raw = pd.read_csv(
    TRAIN_PATH,
    usecols=usecols,
    dtype={'ItemCode': 'category'},
    parse_dates=['Date']
)

for col in float_cols:
    df_raw[col] = (
        df_raw[col].astype(str)
        .str.replace(',', '.', regex=False)
        .astype('float32')
    )

df_raw['Date'] = pd.to_datetime(df_raw['Date'], errors='coerce')
print('shape:', df_raw.shape)
print('date range:', df_raw['Date'].min().date(), '->', df_raw['Date'].max().date())
print(df_raw.dtypes)
df_raw.head(3)


In [ ]:
print('[2] Data audit...')

q = df_raw['Quantity'].astype('float64')
up = df_raw['UnitPrice'].astype('float64')
sa = df_raw['SalesAmount'].astype('float64')
uc = df_raw['Unit Cost'].astype('float64')
ca = df_raw['Cost Amount'].astype('float64')

err_sales = (sa - q * up).abs()
err_cost = (ca - q * uc).abs()

sign_mis_sales = int(((q == 0) & (sa != 0)).sum() + ((q > 0) & (sa < 0)).sum() + ((q < 0) & (sa > 0)).sum())
sign_mis_cost = int(((q == 0) & (ca != 0)).sum() + ((q > 0) & (ca < 0)).sum() + ((q < 0) & (ca > 0)).sum())

print('nulls:', df_raw.isna().sum())
print('UnitPrice <= 0:', int((up <= 0).sum()))
print('Unit Cost <= 0:', int((uc <= 0).sum()))
print('sign mismatch (sales):', sign_mis_sales)
print('sign mismatch (cost):', sign_mis_cost)
print('err_sales quantiles:', err_sales.quantile([0.5, 0.9, 0.95, 0.99]).to_dict())
print('err_cost quantiles:', err_cost.quantile([0.5, 0.9, 0.95, 0.99]).to_dict())

# Conservative anomaly flags (do not overwrite core values)
q_up = up.quantile(0.999)
q_uc = uc.quantile(0.999)
df_raw['flag_bad_price'] = ((df_raw['UnitPrice'] <= 0) | (df_raw['UnitPrice'] > q_up)).astype('int8')
df_raw['flag_bad_cost'] = ((df_raw['Unit Cost'] <= 0) | (df_raw['Unit Cost'] > q_uc)).astype('int8')
print('flag_bad_price:', int(df_raw['flag_bad_price'].sum()), 'flag_bad_cost:', int(df_raw['flag_bad_cost'].sum()))


In [ ]:
print('[3] Aggregate to daily SKU...')

daily = (
    df_raw.groupby(['Date', 'ItemCode'], observed=True, as_index=False)
    .agg({
        'Quantity': 'sum',
        'SalesAmount': 'sum',
        'Cost Amount': 'sum',
        'UnitPrice': 'mean',
        'Unit Cost': 'mean',
        'flag_bad_price': 'max',
        'flag_bad_cost': 'max',
    })
    .rename(columns={'Cost Amount': 'CostAmount'})
)

# Target for training/inference must be non-negative
daily['y'] = daily['Quantity'].clip(lower=0).astype('float32')
daily['profit'] = (daily['SalesAmount'] - daily['CostAmount']).astype('float32')

print('daily shape:', daily.shape)
print('daily negative qty count:', int((daily['Quantity'] < 0).sum()))
print('daily y min:', float(daily['y'].min()))
daily.head(3)


In [ ]:
print('[4] Build complete panel...')
all_dates = pd.date_range(daily['Date'].min(), daily['Date'].max(), freq='D')
all_skus = daily['ItemCode'].cat.categories if str(daily['ItemCode'].dtype) == 'category' else pd.Index(daily['ItemCode'].unique())

panel = pd.MultiIndex.from_product([all_dates, all_skus], names=['Date', 'ItemCode']).to_frame(index=False)
panel['ItemCode'] = panel['ItemCode'].astype('category')
panel = panel.merge(daily, on=['Date', 'ItemCode'], how='left')

fill_zero_cols = ['Quantity', 'SalesAmount', 'CostAmount', 'UnitPrice', 'Unit Cost', 'y', 'profit', 'flag_bad_price', 'flag_bad_cost']
for c in fill_zero_cols:
    panel[c] = panel[c].fillna(0).astype('float32')

print('panel shape:', panel.shape)
print('n_skus:', panel['ItemCode'].nunique(), 'n_days:', panel['Date'].nunique())
panel.head(2)


In [ ]:
def add_calendar_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out['dow'] = out['Date'].dt.dayofweek.astype('int8')
    out['dom'] = out['Date'].dt.day.astype('int8')
    out['month'] = out['Date'].dt.month.astype('int8')
    out['woy'] = out['Date'].dt.isocalendar().week.astype('int16')
    out['is_weekend'] = (out['dow'] >= 5).astype('int8')
    out['is_month_start'] = out['Date'].dt.is_month_start.astype('int8')
    out['is_month_end'] = out['Date'].dt.is_month_end.astype('int8')
    return out


def add_lag_features(df: pd.DataFrame, lags=(1, 7, 14, 28), rolls=(7, 14, 28)) -> pd.DataFrame:
    out = df.copy()
    g = out.groupby('ItemCode', observed=True, sort=False)['y']

    for lag in lags:
        out[f'lag_{lag}'] = g.shift(lag).astype('float32')

    shifted = g.shift(1)
    for w in rolls:
        out[f'rmean_{w}'] = shifted.rolling(w).mean().reset_index(level=0, drop=True).astype('float32')
        out[f'rstd_{w}'] = shifted.rolling(w).std().reset_index(level=0, drop=True).astype('float32')
        out[f'rnz_{w}'] = shifted.rolling(w).apply(lambda x: np.count_nonzero(x > 0), raw=True).reset_index(level=0, drop=True).astype('float32')

    out['price_chg_7'] = (out['UnitPrice'] - out.groupby('ItemCode', observed=True)['UnitPrice'].shift(7)).astype('float32')
    out['cost_chg_7'] = (out['Unit Cost'] - out.groupby('ItemCode', observed=True)['Unit Cost'].shift(7)).astype('float32')
    return out


def make_features(df: pd.DataFrame) -> pd.DataFrame:
    x = add_calendar_features(df)
    x = add_lag_features(x)
    return x


In [ ]:
@dataclass
class WrmsseArtifacts:
    scale: pd.Series
    weights: pd.Series


def build_wrmsse_artifacts(train_panel: pd.DataFrame) -> WrmsseArtifacts:
    diff = train_panel.groupby('ItemCode', observed=True)['y'].diff()
    scale = diff.pow(2).groupby(train_panel['ItemCode']).mean().replace(0, np.nan)

    profit = train_panel.groupby('ItemCode', observed=True)['profit'].sum().clip(lower=0)
    if float(profit.sum()) <= 0:
        weights = pd.Series(1.0 / len(profit), index=profit.index)
    else:
        weights = profit / profit.sum()

    med = float(scale.median()) if np.isfinite(scale.median()) else 1.0
    scale = scale.fillna(med if med > 0 else 1.0)
    return WrmsseArtifacts(scale=scale.astype('float64'), weights=weights.astype('float64'))


def wrmsse_score(y_true, y_pred, sku, art: WrmsseArtifacts) -> float:
    y_true_arr = np.asarray(y_true, dtype=np.float64)
    y_pred_arr = np.asarray(y_pred, dtype=np.float64)
    sku_arr = np.asarray(sku)

    if not (len(y_true_arr) == len(y_pred_arr) == len(sku_arr)):
        raise ValueError('Length mismatch in wrmsse_score inputs')

    df = pd.DataFrame({'sku': sku_arr, 'err2': (y_true_arr - y_pred_arr) ** 2})
    mse_by_sku = df.groupby('sku', observed=True)['err2'].mean()

    common = mse_by_sku.index.intersection(art.scale.index).intersection(art.weights.index)
    if len(common) == 0:
        return np.nan

    rmsse = np.sqrt((mse_by_sku.loc[common] / art.scale.loc[common]).clip(lower=0))
    return float((rmsse * art.weights.loc[common]).sum())


In [ ]:
print('[5] Train/valid split + LGBM Tweedie...')
feat_df = make_features(panel).sort_values(['ItemCode', 'Date']).reset_index(drop=True)
last_date = feat_df['Date'].max()
valid_start = last_date - pd.Timedelta(days=VAL_DAYS - 1)

feature_cols = [
    c for c in feat_df.columns
    if c not in ['Date', 'ItemCode', 'Quantity', 'SalesAmount', 'CostAmount', 'UnitPrice', 'Unit Cost', 'profit', 'y']
]

train_df = feat_df[feat_df['Date'] < valid_start].copy()
valid_df = feat_df[feat_df['Date'] >= valid_start].copy()

# Controlled fill for model matrix only
X_train = train_df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)
X_valid = valid_df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)
y_train = train_df['y'].values
y_valid = valid_df['y'].values

wr_art = build_wrmsse_artifacts(train_df[['ItemCode', 'y', 'profit']].copy())

params = dict(
    objective='tweedie',
    tweedie_variance_power=1.2,
    learning_rate=0.05,
    n_estimators=350 if FAST_MODE else 650,
    num_leaves=127,
    min_child_samples=80,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.05,
    reg_lambda=1.0,
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1,
)

model = lgb.LGBMRegressor(**params)
model.fit(X_train, y_train)

pred_valid = np.clip(model.predict(X_valid), 0, None)
rmse = float(np.sqrt(mean_squared_error(y_valid, pred_valid)))
wr = wrmsse_score(y_valid, pred_valid, valid_df['ItemCode'], wr_art)

print('valid RMSE:', rmse)
print('valid WRMSSE:', wr)


In [ ]:
print('[6] Refit on full history...')
full_feat = make_features(panel).sort_values(['ItemCode', 'Date']).reset_index(drop=True)
X_full = full_feat[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)
y_full = full_feat['y'].values

model_full = lgb.LGBMRegressor(**params)
model_full.fit(X_full, y_full)
print('full fit done:', len(full_feat))


In [ ]:
print('[7] Recursive 56-day forecast (stable state update)...')
state = panel[['Date', 'ItemCode', 'Quantity', 'SalesAmount', 'CostAmount', 'UnitPrice', 'Unit Cost', 'profit', 'y', 'flag_bad_price', 'flag_bad_cost']].copy()
state = state.sort_values(['ItemCode', 'Date']).reset_index(drop=True)

all_skus = state['ItemCode'].cat.categories if str(state['ItemCode'].dtype) == 'category' else pd.Index(state['ItemCode'].unique())
last_date = state['Date'].max()
future_dates = pd.date_range(last_date + pd.Timedelta(days=1), periods=HORIZON, freq='D')

pred_rows = []
for d in future_dates:
    step = pd.DataFrame({'Date': d, 'ItemCode': all_skus})
    step['ItemCode'] = step['ItemCode'].astype('category')

    # Keep static/value columns neutral for future rows
    step['Quantity'] = 0.0
    step['SalesAmount'] = 0.0
    step['CostAmount'] = 0.0
    step['UnitPrice'] = 0.0
    step['Unit Cost'] = 0.0
    step['profit'] = 0.0
    step['flag_bad_price'] = 0.0
    step['flag_bad_cost'] = 0.0
    step['y'] = np.nan

    state = pd.concat([state, step], ignore_index=True)
    state = state.sort_values(['ItemCode', 'Date']).reset_index(drop=True)

    feat_state = make_features(state)
    cur = feat_state[feat_state['Date'] == d].copy()
    X_cur = cur[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)

    pred = np.clip(model_full.predict(X_cur), 0, None)
    state.loc[state['Date'] == d, 'y'] = pred

    pred_rows.append(pd.DataFrame({'Date': d, 'ItemCode': cur['ItemCode'].values, 'pred': pred}))

pred56 = pd.concat(pred_rows, ignore_index=True)
print('pred56 shape:', pred56.shape)
print('pred quantiles:', pred56['pred'].quantile([0, 0.5, 0.9, 0.99]).to_dict())
print('share pred < 1e-6:', float((pred56['pred'] < 1e-6).mean()))


In [ ]:
print('[8] Build submission_v1.csv...')
sub_template = pd.read_csv(SUB_PATH)

sku_to_vals = (
    pred56.sort_values(['ItemCode', 'Date'])
    .groupby('ItemCode', observed=True)['pred']
    .apply(list)
    .to_dict()
)

fcols = [f'F{i}' for i in range(1, 29)]
rows = []
for rid in sub_template['id']:
    sku, suffix = rid.rsplit('_', 1)
    vals = sku_to_vals.get(sku, [0.0] * HORIZON)
    block = vals[:28] if suffix == 'validation' else vals[28:56]
    rows.append([rid] + [float(max(0.0, v)) for v in block])

sub = pd.DataFrame(rows, columns=['id'] + fcols)

assert sub.shape == sub_template.shape
assert sub['id'].is_unique
assert set(sub['id']) == set(sub_template['id'])
assert np.isfinite(sub[fcols].to_numpy()).all()
assert (sub[fcols].to_numpy() >= 0).all()

sub.to_csv('submission_v1.csv', index=False)
print('saved submission_v1.csv', sub.shape)
sub.head(3)


In [ ]:
print('[9] Diagnostics table for top weighted SKUs...')
profit_sku = panel.groupby('ItemCode', observed=True)['profit'].sum().clip(lower=0)
if float(profit_sku.sum()) > 0:
    w = profit_sku / profit_sku.sum()
else:
    w = pd.Series(1.0 / len(profit_sku), index=profit_sku.index)

top = w.sort_values(ascending=False).head(20).index
pred_stats = pred56[pred56['ItemCode'].isin(top)].groupby('ItemCode', observed=True)['pred'].agg(['mean','median','max'])
out = pred_stats.join(w.rename('weight'), how='left').sort_values('weight', ascending=False)
out
